In [1]:
"""
Adapted from: https://robosuite.ai/docs/modules/environments.html
"""
import math
import os
import sys

import numpy as np
import mediapy as media

import robosuite
from robosuite.controllers import load_composite_controller_config

from openpi.models import model as _model
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config

sys.path.append('../py_script')
import profiling as prof
prof.prof_start()

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /storage/ice1/3/8/jpeng303/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


In [2]:
def _quat2axisangle(quat):
    """
    Copied from robosuite: https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py#L490C1-L512C55
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

In [3]:
# Joint control
# controller_file = os.path.join(os.path.dirname(__file__), "..", "py_script", "panda_joint_controller.json")
# controller_config = load_composite_controller_config(controller=controller_file)
# Load pi model
# vla_config = _config.get_config("pi05_droid")
# checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi05_droid")
# @prof.profiled
# def prompt_from_obs(obs):
#    qpos = np.array(obs['robot0_joint_pos'])
#    gripper_pos = np.array(obs['robot0_gripper_qpos'][0])
#    return {
#        # Flip the ego camera?
#        'observation/exterior_image_1_left': obs['agentview_image'][::-1, ::-1, :],
#        'observation/wrist_image_left': obs['robot0_eye_in_hand_image'][::-1, ::-1, :],
#        'observation/joint_position': qpos,
#        'observation/gripper_position': gripper_pos,
#        'prompt': 'Grab and pick up the red cube'
#    }

In [4]:
# Pose control
controller_config = load_composite_controller_config(controller="BASIC")
# Load pi model
vla_config = _config.get_config("pi05_libero")
checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi05_base")
@prof.profiled
def prompt_from_obs(obs):
    # Reference: https://github.com/Physical-Intelligence/openpi/blob/981483dca0fd9acba698fea00aa6e52d56a66c58/examples/libero/main.py#L130
    return {
        # Flip the ego camera?
        'observation/image': obs['agentview_image'][::-1, ::-1, :],
        'observation/wrist_image': obs['robot0_eye_in_hand_image'][::-1, ::-1, :],
        'observation/state': np.concatenate(
            (
                obs["robot0_eef_pos"],
                _quat2axisangle(obs["robot0_eef_quat"]),
                obs["robot0_gripper_qpos"],
            )
        ),
        'prompt': 'Stack the red cube onto the green cube'
    }

[robosuite INFO] Loading controller configuration from: /storage/ice1/3/8/jpeng303/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
INFO:robosuite_logs:Loading controller configuration from: /storage/ice1/3/8/jpeng303/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/controllers/config/default/composite/basic.json


In [5]:
# Create a trained policy.
policy = _policy_config.create_trained_policy(vla_config, checkpoint_dir)

NameError: name 'env' is not defined

In [6]:
freq = 20
episode_length=800
env = robosuite.make(
    "Stack",
    robots=["Panda"],
    gripper_types="default",
    controller_configs=controller_config,
    env_configuration="opposed",    # What are the options?
    has_renderer=False,
    #render_camera="frontview",
    has_offscreen_renderer=True,
    control_freq=freq,
    horizon=episode_length,
    use_object_obs=False,
    use_camera_obs=True,
    camera_names=["agentview", "robot0_eye_in_hand"],
    camera_heights=224,
    camera_widths=224,
)
env_step = prof.profiled(env.step, name="step")

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

In [11]:
rollout = []
frames = []
wrist_frames = []
act_scale = 0.5
for _ in range(1):
    obs = env.reset()
    prompt = prompt_from_obs(obs)
    vla_output = policy.infer(prompt)
    actions = vla_output['actions']
    #print(vla_output['subtask'])
    rollout.append(obs)

    policy_infer = prof.profiled(policy.infer, name="infer")

    trajectory_idx = 0
    for i in range(episode_length):
        act = np.copy(actions[trajectory_idx])
        act[-1] *= 1
        obs, reward, done, info = env_step(act * act_scale)
        rollout.append(obs)
        frames.append(obs['agentview_image'][::-1, ::-1, :])
        wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])
        trajectory_idx += 1
        if trajectory_idx == (len(actions)//2):
            prompt = prompt_from_obs(obs)
            vla_output = policy_infer(prompt)
            actions = vla_output['actions']
            #print(vla_output['subtask'], flush=True)
            trajectory_idx = 0

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up green lego
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up green lego
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up greenⓍ javier
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up green lid
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue lid
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue sparkling water
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue lid
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue sparkling water
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue lid
tokenize high prompt: Stack the red cube onto the green cube
Generated Subtask: pick up blue 

In [12]:
media.write_video('franka_stacking.mp4', frames, fps=freq)
media.write_video('franka_stacking_wrist.mp4', wrist_frames, fps=freq)